In [1]:
# Imports
import librosa
import torch
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from transformers import AutoProcessor, VoxtralRealtimeForConditionalGeneration

# Dispositivo = GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

Ejecutando en: cuda:0


In [2]:
# Cargar modelo y procesador 
repo_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"
print(f"Cargando modelo y procesador: {repo_id}")

processor = AutoProcessor.from_pretrained(repo_id)
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(repo_id).to(device)
model.eval()

Cargando modelo y procesador: mistralai/Voxtral-Mini-4B-Realtime-2602


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

VoxtralRealtimeForConditionalGeneration(
  (audio_tower): VoxtralRealtimeEncoder(
    (embedder): VoxtralRealtimeEmbedder(
      (conv1): VoxtralRealtimeCausalConv1d(128, 1280, kernel_size=(3,), stride=(1,))
      (conv2): VoxtralRealtimeCausalConv1d(1280, 1280, kernel_size=(3,), stride=(2,))
    )
    (layers): ModuleList(
      (0-31): 32 x VoxtralRealtimeEncoderLayer(
        (self_attn): VoxtralRealtimeAttention(
          (q_proj): Linear(in_features=1280, out_features=2048, bias=True)
          (k_proj): Linear(in_features=1280, out_features=2048, bias=False)
          (v_proj): Linear(in_features=1280, out_features=2048, bias=True)
          (o_proj): Linear(in_features=2048, out_features=1280, bias=True)
        )
        (self_attn_layer_norm): VoxtralRealtimeRMSNorm((1280,), eps=1e-05)
        (activation_fn): GELUActivation()
        (final_layer_norm): VoxtralRealtimeRMSNorm((1280,), eps=1e-05)
        (mlp): VoxtralRealtimeMLP(
          (gate_proj): Linear(in_features=128

In [ ]:
audios = []

lst_path = r"..\..\Europarl-ST\en\es\test\Europarl-ST.v2.en.es.test.lst"

with open(lst_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        if linea.strip():
            audios.append(str(linea).strip())
        
print(f"Total de audios a procesar: {len(audios)}")

Total de audios a procesar: 127


In [ ]:
hypotheses = []
references = []

for audio in tqdm(audios, desc="Transcribiendo con Voxtral (inglés)"):
    
    ref_path = r"..\..\Europarl-ST\en\es\test\%s\transcription.tok" % audio
    with open(ref_path, "r", encoding="utf-8") as referencia:
        references.append(referencia.read())

    audio_path = r"..\..\Europarl-ST\en\es\test\%s\audio_clip_diarization.m4a" % audio
    audio_array, sr = librosa.load(audio_path, sr=16000)
    
    inputs = processor(audio_array, return_tensors="pt").to(device)
    if "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(model.dtype)
    
    with torch.no_grad():
        outputs = model.generate(**inputs)
        
    decoded_text = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    hypotheses.append(decoded_text)

Transcribiendo con Voxtral (inglés):   0%|          | 0/127 [00:00<?, ?it/s]

C:\Users\carru\AppData\Local\Temp\ipykernel_34476\1581257341.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sr = librosa.load(audio_path, sr=16000)
d:\carru\voxtral-tfg\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
d:\carru\voxtral-tfg\venv\Lib\site-packages\transformers\generation\utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=1037) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
C:\Users\carru\AppData\Local\Temp\ipykernel_34476\1581257341.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sr = librosa.load(audio_path, sr=16000)
d:\carru\voxtral-tfg\venv\Lib\site-packages\librosa\core\audio.py:184: Futu

In [5]:
# Crear DataFrame y guardar resultados
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references))

data.to_csv("voxtral_raw_results_en.csv", index=False)

data.head()

,hypothesis,reference
0,"His only greed, euphoria and cheap money to b...","Madam President , are only greed , euphoria an..."
1,The financial crisis rages on and Eurozone co...,"Madam President , the financial crisis rages o..."
2,ISIS has demonstrated that the global financi...,"Mr President , this crisis has demonstrated th..."
3,Thank you. They're going to be radical change...,"Mr President , there are going to be radical c..."
4,"Thank you, Madam Chair. Sometimes questions f...","Sometimes a question flies away , and this is ..."


In [6]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(text)) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(text)) for text in data["reference"]]

data.to_csv("voxtral_resultados_limpios_en.csv", index=False)

data.head()

,hypothesis,reference,hypothesis_clean,reference_clean
0,"His only greed, euphoria and cheap money to b...","Madam President , are only greed , euphoria an...",his only greed euphoria and cheap money to be...,madam president are only greed euphoria and ch...
1,The financial crisis rages on and Eurozone co...,"Madam President , the financial crisis rages o...",the financial crisis rages on and eurozone co...,madam president the financial crisis rages on ...
2,ISIS has demonstrated that the global financi...,"Mr President , this crisis has demonstrated th...",isis has demonstrated that the global financi...,mr president this crisis has demonstrated that...
3,Thank you. They're going to be radical change...,"Mr President , there are going to be radical c...",thank you they re going to be radical changes...,mr president there are going to be radical cha...
4,"Thank you, Madam Chair. Sometimes questions f...","Sometimes a question flies away , and this is ...",thank you madam chair sometimes questions fly...,sometimes a question flies away and this is pe...


In [7]:
# Cálculo del Word Error Rate (WER)
wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))

print(f"WER final de Voxtral: {wer * 100:.2f} %")

WER final de Voxtral: 15.35 %


WER obtenido: 15.35%